In [ ]:
import json
import numpy as np
import shutil
import pandas as pd

In [ ]:
df = pd.read_csv('../mhc_data/TCR3d_data.csv')
df['PDB ID'] = df['PDB ID'].astype(str).str.lower().str.strip()
df['Release date'] = pd.to_datetime(df['Release date'])

In [ ]:
mask_train = df['Release date'] < '2021-09-30'
train_df = df[mask_train]

In [5]:
len(train_df)

1082

In [6]:
mask_val = (df['Release date'] >= '2021-09-30') & (df['Release date'] < '2023-01-13')
val_df = df[mask_val]

In [7]:
len(val_df)

130

In [8]:
train_ids = train_df['PDB ID'].dropna().tolist()
val_ids = val_df['PDB ID'].dropna().tolist()

In [18]:
# Apenas para guardar essa informação, não é necessário rodar esse código novamente
# No resto do código vamos usar o train_ids e val_ids

with open('../mhc_data/train_templated.txt', 'w') as f:
    f.write('\n'.join(train_ids))

with open('../mhc_data/val_templated.txt', 'w') as f:
    f.write('\n'.join(val_ids))

In [9]:
with open('../rcsb_processed_targets/manifest.json') as f:
    data = json.load(f)

In [22]:
with open('../mhc_data/train_templated.json') as f:
    train = json.load(f)

with open('../mhc_data/val_templated.json') as f:
    val = json.load(f)

In [ ]:
# train_val = [sample for sample in train_val if sample['PDB ID']=='1a1m']

In [25]:
train_val = train + val

In [24]:
manifest_dict = {sample['id']: sample for sample in data}

In [20]:
import os
import urllib.request
import glob
import gemmi

def baixar_cif_do_pdb(pdb_id, diretorio_destino="../cif_files"):
    """
    Baixa o arquivo .cif do banco de dados RCSB PDB.
    """
    os.makedirs(diretorio_destino, exist_ok=True)
    
    # O RCSB PDB fornece arquivos CIF através desta URL padrão
    url = f"https://files.rcsb.org/download/{pdb_id.lower()}.cif"
    caminho_arquivo = os.path.join(diretorio_destino, f"{pdb_id.lower()}.cif")
    
    # Baixa apenas se o arquivo ainda não existir localmente
    if not os.path.exists(caminho_arquivo):
        try:
            urllib.request.urlretrieve(url, caminho_arquivo)
            print(f"Sucesso ao baixar: {pdb_id.upper()}")
        except Exception as e:
            print(f"Erro ao baixar {pdb_id.upper()}: {e}")
            return None
    else:
        print(f"Arquivo já existe: {pdb_id.upper()}")
            
    return caminho_arquivo

def processar_dataset_npz(diretorio_npz):
    """
    Lê os arquivos .npz, baixa os .cif correspondentes e aplica a heurística do MHC.
    """
    # Busca todos os arquivos .npz no diretório descompactado
    arquivos_npz = glob.glob(os.path.join(diretorio_npz, "*.npz"))
    
    resultados_finais = {}

    for caminho_npz in arquivos_npz:
        nome_arquivo = os.path.basename(caminho_npz)
        
        # Extrai os primeiros 4 caracteres, que geralmente representam o código PDB
        pdb_id = nome_arquivo[:4] 
        
        # 1. Baixa o arquivo CIF original
        # caminho_cif = baixar_cif_do_pdb(pdb_id)
        baixar_cif_do_pdb(pdb_id)
        
        # if caminho_cif:
        #     # 2. Roda a função do gemmi (definida anteriormente)
        #     # Para o contexto de limpeza de dados, esta etapa permite mapear
        #     # exatamente quais cadeias no .npz correspondem a quais moléculas
        #     classificacao = classificar_complexo_mhc(caminho_cif) 
        #     resultados_finais[pdb_id] = classificacao
            
    return resultados_finais

# Código da função classificar_complexo_mhc definida na resposta anterior
def classificar_complexo_mhc(caminho_arquivo):
    estrutura = gemmi.read_structure(caminho_arquivo)
    modelo = estrutura[0]
    print(modelo)
    
    mhc_chains, b2m_chains, peptide_chains = [], [], []

    for chain in modelo:
        polimero = chain.get_polymer()
        if polimero.check_polymer_type() != gemmi.PolymerType.PeptideL:
            continue
            
        sequencia = polimero.make_one_letter_sequence()
        tamanho = len(sequencia)
        
        if 8 <= tamanho <= 25:
            peptide_chains.append(chain.name)
        elif tamanho > 150:
            mhc_chains.append(chain.name)
        elif 80 <= tamanho <= 120:
            b2m_chains.append(chain.name)

    # return {"MHC": mhc_chains, "B2M": b2m_chains, "Peptideos": peptide_chains}

# Exemplo de uso
# processar_dataset_npz("../rcsb_processed_targets/structures/")
# classificar_complexo_mhc("./cif_files/1a1m.cif")
with open('../mhc_data/val_templated.txt', 'r') as f:
    trains = [line.strip() for line in f]

print(f"Total de amostras de treino: {len(trains)}")

for i, id in enumerate(trains):
    caminho_cif = baixar_cif_do_pdb(id)
    print(f"{i+1}/{len(trains)} - {id} - {caminho_cif}")
    


Total de amostras de treino: 130
Sucesso ao baixar: 6ZKW
1/130 - 6zkw - ../cif_files/6zkw.cif
Sucesso ao baixar: 6ZKX
2/130 - 6zkx - ../cif_files/6zkx.cif
Sucesso ao baixar: 6ZKY
3/130 - 6zky - ../cif_files/6zky.cif
Sucesso ao baixar: 6ZKZ
4/130 - 6zkz - ../cif_files/6zkz.cif
Arquivo já existe: 7BBG
5/130 - 7bbg - ../cif_files/7bbg.cif
Sucesso ao baixar: 7BH8
6/130 - 7bh8 - ../cif_files/7bh8.cif
Sucesso ao baixar: 7CIQ
7/130 - 7ciq - ../cif_files/7ciq.cif
Sucesso ao baixar: 7CIR
8/130 - 7cir - ../cif_files/7cir.cif
Sucesso ao baixar: 7CIS
9/130 - 7cis - ../cif_files/7cis.cif
Sucesso ao baixar: 7DUU
10/130 - 7duu - ../cif_files/7duu.cif
Arquivo já existe: 7DYN
11/130 - 7dyn - ../cif_files/7dyn.cif
Sucesso ao baixar: 7DZM
12/130 - 7dzm - ../cif_files/7dzm.cif
Sucesso ao baixar: 7DZN
13/130 - 7dzn - ../cif_files/7dzn.cif
Sucesso ao baixar: 7EJL
14/130 - 7ejl - ../cif_files/7ejl.cif
Sucesso ao baixar: 7EJM
15/130 - 7ejm - ../cif_files/7ejm.cif
Sucesso ao baixar: 7EJN
16/130 - 7ejn - ../cif

In [6]:
import os
import glob
import json
import gemmi
from Bio import Align

# ---------------------------------------------------------
# Funções Biológicas e Geométricas
# ---------------------------------------------------------
def extrair_sequencia(chain):
    polimero = chain.get_polymer()
    if polimero.check_polymer_type() == gemmi.PolymerType.PeptideL:
        return polimero.make_one_letter_sequence()
    return ""

def identificar_mhc_por_alinhamento(modelo_alvo, sequencia_ref_1a1m):
    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    melhor_cadeia = None
    maior_score = -1
    
    for chain in modelo_alvo:
        seq_alvo = extrair_sequencia(chain)
        if len(seq_alvo) < 100: 
            continue
        score = aligner.score(sequencia_ref_1a1m, seq_alvo)
        if score > maior_score:
            maior_score = score
            melhor_cadeia = chain
    return melhor_cadeia

def calcular_distancia_minima(chain_a, chain_b):
    min_dist = float('inf')
    pos_a = [res.find_atom('CA', '*').pos for res in chain_a.get_polymer() if res.find_atom('CA', '*')]
    pos_b = [res.find_atom('CA', '*').pos for res in chain_b.get_polymer() if res.find_atom('CA', '*')]
    
    for pa in pos_a:
        for pb in pos_b:
            dist = pa.dist(pb)
            if dist < min_dist:
                min_dist = dist
    return min_dist

def extrair_complexo_formato_boltz(caminho_alvo, seq_1a1m, pdb_id):
    """
    Realiza a validação e retorna a estrutura EXATA esperada pelo Notebook.
    """
    try:
        est_alvo = gemmi.read_structure(caminho_alvo)
        mod_alvo = est_alvo[0]
    except Exception as e:
        return None, f"Erro de leitura: {str(e)}"
    
    # 1. MHC via Sequência
    mhc_alvo_chain = identificar_mhc_por_alinhamento(mod_alvo, seq_1a1m)
    if not mhc_alvo_chain:
        return None, "MHC não encontrado."
        
    # 2. Peptídeo via Proximidade Geométrica
    melhor_peptideo = None
    menor_distancia = float('inf')
    
    for chain in mod_alvo:
        if chain.name == mhc_alvo_chain.name:
            continue
            
        seq_pep = extrair_sequencia(chain)
        if 8 <= len(seq_pep) <= 25:
            dist = calcular_distancia_minima(mhc_alvo_chain, chain)
            if dist < menor_distancia:
                menor_distancia = dist
                melhor_peptideo = chain
                
    if menor_distancia > 5.0 or not melhor_peptideo:
        return None, "Peptídeo fora da fenda de ligação."
        
    # --- A MÁGICA ACONTECE AQUI ---
    # Montamos o dicionário do jeito que o Jupyter Notebook espera ler
    amostra_formatada = {
        "pdb_id": pdb_id.lower(),             # ex: "1a1m"
        "peptide_chain": melhor_peptideo.name, # ex: "C" (o notebook vai somar '1' e virar 'C1')
        "protein_chains": [mhc_alvo_chain.name] # ex: ["A"] (o notebook vai pegar o índice [0] e somar '1')
    }
    
    return amostra_formatada, "Sucesso"

# ---------------------------------------------------------
# Processamento em Lote e Exportação
# ---------------------------------------------------------
def gerar_json_treinamento(diretorio_cifs, caminho_1a1m, arquivo_saida="train_templated.json"):
    print(f"Lendo PDBs em: {diretorio_cifs}")
    
    # Carrega a referência
    est_ref = gemmi.read_structure(caminho_1a1m)
    seq_1a1m = extrair_sequencia(est_ref[0]['A'])
    
    arquivos_cif = glob.glob(os.path.join(diretorio_cifs, "*.cif"))
    
    dataset_limpo = []
    erros = 0
    
    for caminho_alvo in arquivos_cif:
        nome_arquivo = os.path.basename(caminho_alvo)
        pdb_id = nome_arquivo.replace('.cif', '').lower()
        
        amostra, log = extrair_complexo_formato_boltz(caminho_alvo, seq_1a1m, pdb_id)
        
        if amostra:
            dataset_limpo.append(amostra)
        else:
            # print(f"Descartado {pdb_id}: {log}") # Descomente se quiser ver o motivo de cada descarte
            erros += 1
            
    # Salva o arquivo EXATAMENTE como o notebook espera (uma lista de objetos)
    with open(arquivo_saida, 'w') as f:
        json.dump(dataset_limpo, f, indent=4)
        
    print(f"\n--- Concluído ---")
    print(f"Complexos Válidos (Limpos): {len(dataset_limpo)}")
    print(f"Descartados (Ruído): {erros}")
    print(f"Arquivo gerado: {arquivo_saida}")

# Execução
gerar_json_treinamento("./cif_files", "./cif_files/1a1m.cif", "meu_train_templated.json")

Lendo PDBs em: ./cif_files


KeyboardInterrupt: 

In [30]:
target_dir = 'mhc_samples'

import os
os.makedirs(target_dir, exist_ok=True)
os.makedirs(f"{target_dir}/msa/", exist_ok=True)
os.makedirs(f"{target_dir}/structures/", exist_ok=True)

new_manifest = []

bad_ids = []

for sample in train_val:
    if sample['pdb_id'] not in manifest_dict:
        bad_ids.append(sample['pdb_id'])
        continue
    sample_manifest = manifest_dict[sample['pdb_id']]
    peptide_chain_name = sample['peptide_chain'] + '1'
    protein_chain_name = sample['protein_chains'][0] + '1'
    matched_chains = 0
    valid_chain_ids = []
    for chain in sample_manifest['chains']:
        if chain['msa_id'] != -1:
            shutil.copy(f"../rcsb_processed_msa/{chain['msa_id']}.npz", f"{target_dir}/msa/{chain['msa_id']}.npz")
        if chain['chain_name'] in [peptide_chain_name, protein_chain_name]:
            matched_chains += 1
            valid_chain_ids.append(chain['chain_id'])
        else:
            chain['valid'] = False
    if matched_chains != 2:
        bad_ids.append(sample['pdb_id'])
        # print('Didnt find both chains', sample['PDB ID'])
    else:
        n_correct_interfaces = 0
        for interface in sample_manifest['interfaces']:
            if (interface['chain_1'] in valid_chain_ids) and (interface['chain_2'] in valid_chain_ids):
                n_correct_interfaces += 1
            else:
                interface['valid'] = False
        if n_correct_interfaces != 1:
            print('Number of correct interfaces is wrong:', sample['pdb_id'], n_correct_interfaces)
        else:
            new_manifest.append(sample_manifest)
            
            # update mask in npz just in case
            npz = dict(np.load(f"../rcsb_processed_targets/structures/{sample_manifest['id']}.npz"))
            for chain_id in range(len(npz['mask'])):
                if chain_id in valid_chain_ids:
                    npz['mask'][chain_id] = True
                else:
                    npz['mask'][chain_id] = False
            np.savez(f"{target_dir}/structures/{sample_manifest['id']}", **npz)
    # break

Number of correct interfaces is wrong: 5tez 0
Number of correct interfaces is wrong: 5trz 0
Number of correct interfaces is wrong: 6gh1 0
Number of correct interfaces is wrong: 7bh8 0
Number of correct interfaces is wrong: 7n2n 0
Number of correct interfaces is wrong: 7n2o 0
Number of correct interfaces is wrong: 7n2s 0
Number of correct interfaces is wrong: 7phr 0
Number of correct interfaces is wrong: 7s7d 0
Number of correct interfaces is wrong: 8d5q 0
Number of correct interfaces is wrong: 5tez 0
Number of correct interfaces is wrong: 5trz 0
Number of correct interfaces is wrong: 6gh1 0
Number of correct interfaces is wrong: 5tez 0
Number of correct interfaces is wrong: 5trz 0
Number of correct interfaces is wrong: 6gh1 0
Number of correct interfaces is wrong: 7bh8 0
Number of correct interfaces is wrong: 7n2n 0
Number of correct interfaces is wrong: 7n2o 0
Number of correct interfaces is wrong: 7n2s 0
Number of correct interfaces is wrong: 7phr 0
Number of correct interfaces is wr

In [31]:
with open(f"{target_dir}/manifest_samples.json", "w") as outfile:
    outfile.write(json.dumps(new_manifest))

In [33]:
new_manifest

[{'id': '1a1m',
  'structure': {'resolution': 0.0,
   'method': 'x-ray diffraction',
   'deposited': '1997-12-11',
   'released': '1998-04-08',
   'revised': '2023-08-02',
   'num_chains': 3,
   'num_interfaces': 2},
  'chains': [{'chain_id': 0,
    'chain_name': 'A1',
    'mol_type': 0,
    'cluster_id': '2yez_0',
    'msa_id': '1a1m_a',
    'template_id': '1a1m_a',
    'num_residues': 278,
    'valid': True},
   {'chain_id': 1,
    'chain_name': 'B1',
    'mol_type': 0,
    'cluster_id': '7cjq_1',
    'msa_id': '7kgr_b',
    'template_id': '7kgr_b',
    'num_residues': 99,
    'valid': False},
   {'chain_id': 2,
    'chain_name': 'C1',
    'mol_type': 0,
    'cluster_id': '1a1m_2',
    'msa_id': '1a1m_c',
    'template_id': '1a1m_c',
    'num_residues': 9,
    'valid': True}],
  'interfaces': [{'chain_1': 0, 'chain_2': 1, 'valid': False},
   {'chain_1': 0, 'chain_2': 2, 'valid': True}],
  'affinity': None,
  'md': None},
 {'id': '1a1n',
  'structure': {'resolution': 0.0,
   'method':

In [32]:
len(train_val), len(bad_ids)

(4521, 1429)

In [98]:
val_list = [sample['pdb_id'].upper() for sample in val]

with open("{target_dir}/validation_ids.txt", "w") as outfile:
    outfile.write('\n'.join(val_list))

In [118]:
# !zip -r mhc.zip mhc_targets/

In [115]:
# scp mhc.zip eglukhov@nabu5.ams.stonybrook.edu:/home/eglukhov/projects/boltz/train_data/

In [ ]:
< 2021-09-30 - train
< 2023-01-13 - val
> - test